In [3]:
import pandas as pd

# Die Datei einlesen
spaltennamen = ['qid', 'iter', 'docid', 'rel']
df = pd.read_csv(r"C:\Users\muham\Downloads\qrels.txt", sep=r'\s+', names=spaltennamen, header=None)

# Nur die Zeilen behalten, bei denen die Relevanz 0 ist
df_nicht_relevant = df[df['rel'] == 0]

# Ein kurzer Kontrollblick in die gefilterten Daten
print(df_nicht_relevant.head())

                                qid  iter      docid  rel
0  10c7ef19d812f4c1bacecc0ee54d30a8     0   10557343    0
1  10c7ef19d812f4c1bacecc0ee54d30a8     0   10962698    0
2  10c7ef19d812f4c1bacecc0ee54d30a8     0  126830649    0
3  10c7ef19d812f4c1bacecc0ee54d30a8     0  148700944    0
4  10c7ef19d812f4c1bacecc0ee54d30a8     0  160926486    0


In [19]:
print(f"Nicht relevante Dokumente: {len(df_nicht_relevant)}") 
# Kommt hin wenn wir 30 Topics mit ca. 50 docs pro topic haben und 30-40% n. rel sind

Nicht relevante Dokumente: 688


In [9]:
import pandas as pd
import glob

# 1. Schritt: Alle 12 JSONL-Dateien einlesen und zu einem großen DataFrame verbinden
# glob.glob findet alle .jsonl Dateien in dem angegebenen Ordner
jsonl_dateien = glob.glob(r"C:\Users\muham\Desktop\longeval_sci_test-09-11_2026_abstract\data\processed\doc_collection_09032026_abstract_2\snapshot-3\longeval_sci_test-09-11_2026_abstract\documents/*.jsonl") 

korpus_listen = []
for datei in jsonl_dateien:
    temporaeres_df = pd.read_json(datei, lines=True)
    korpus_listen.append(temporaeres_df)

# Alle einzelnen DataFrames zu einem großen Korpus zusammenfügen
df_volltext_korpus = pd.concat(korpus_listen, ignore_index=True)

# Sicherstellen, dass die Spaltennamen passen 
print(f"Gesamtanzahl der Texte im Korpus: {len(df_volltext_korpus)}")


# 2. Schritt: Verknüpfen mit den unterschiedlichen Spaltennamen
# left_on bezieht sich auf df_nicht_relevant (docid)
# right_on bezieht sich auf df_volltext_korpus (id)
df_analyse_pool = pd.merge(
    df_nicht_relevant, 
    df_volltext_korpus, 
    left_on='docid', 
    right_on='id', 
    how='inner'
)

# Die doppelte ID-Spalte 'id' entfernen, da wir 'docid' behalten
df_analyse_pool = df_analyse_pool.drop(columns=['id'])

print(f"Anzahl der irrelevanten Dokumente mit verfügbarem Volltext: {len(df_analyse_pool)}")

Gesamtanzahl der Texte im Korpus: 1661900
Anzahl der irrelevanten Dokumente mit verfügbarem Volltext: 688


In [13]:
!pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     -------------------------------------- 981.5/981.5 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993251 sha256=00ab5e638ec4cce11aa922c15e0c434db36445463b8ad16cd45502ecdf7be2c1
  Stored in directory: c:\users\muham\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [15]:
from langdetect import detect, LangDetectException

# 1. Die Erkennungsfunktion definieren
def ist_fremdsprachig_langdetect(text):
    # Falls das Abstract fehlt (NaN) oder leer ist, wird es übersprungen (False)
    if not isinstance(text, str) or not text.strip():
        return False
    
    try:
        # Erkennt die Sprache 
        sprache = detect(text)
        # Wenn es NICHT Englisch ('en') ist, geben wir True zurück
        return sprache != 'en'
    except LangDetectException:
        # Falls der Text nur aus Zahlen oder Sonderzeichen besteht
        return False

# 2. Die Funktion auf die Spalte 'abstract' anwenden
print("Starte Spracherkennung basierend auf den Abstracts...")
df_analyse_pool['ist_fremdsprachig'] = df_analyse_pool['abstract'].apply(ist_fremdsprachig_langdetect)

# 3. Filtern: Nur die Dokumente behalten, die fremdsprachig sind 
df_relabel_liste = df_analyse_pool[df_analyse_pool['ist_fremdsprachig'] == True]

# 4. Ergebnis ausgeben
print("--- Analyse abgeschlossen ---")
print(f"Gefundene fremdsprachige Dokumente für das Relabeling: {len(df_relabel_liste)}")

Starte Spracherkennung basierend auf den Abstracts... Einen Moment bitte.
--- Analyse abgeschlossen ---
Gefundene fremdsprachige Dokumente für das Relabeling: 92


In [17]:
# Speichert die IDs, ohne Index oder Spaltennamen, zeilenweise ab
df_relabel_liste['docid'].to_csv('relabel_doc_ids.txt', index=False, header=False)
print("Die Dokumenten-IDs wurden gespeichert.")

Die Dokumenten-IDs wurden gespeichert.


In [21]:
# Ergebnisse als csv speochern
print("\n--Analyse abgeschlossen")
print(f"Gefundene fremdsprachige Dokumente für das Relabeling: {len(df_relabel_liste)}")

# Exportiert qid, docid, title und abstract in eine CSV-Datei.
df_relabel_liste[['qid', 'docid', 'title', 'abstract']].to_csv(
    'fremdsprachige_dokumente_volltext.csv', 
    index=False, 
    sep=';', 
    encoding='utf-8-sig'
)

print("\nDie Datei 'fremdsprachige_dokumente_volltext.csv' wurde erfolgreich erstellt.")


--Analyse abgeschlossen
Gefundene fremdsprachige Dokumente für das Relabeling: 92

Die Datei 'fremdsprachige_dokumente_volltext.csv' wurde erfolgreich erstellt.


In [23]:
import numpy as np

# 1. Das Dataframe mit den relevanten Spalten filtern
df_export = df_relabel_liste[['qid', 'docid', 'title', 'abstract']]

# 2. Das Dataframe mit NumPy in 3 möglichst gleich große Teile splitten
teil_1, teil_2, teil_3 = np.array_split(df_export, 3)

# 3. Die drei Teile als separate CSV-Dateien für dein Team abspeichern
einstellungen = {'index': False, 'sep': ';', 'encoding': 'utf-8-sig'}

teil_1.to_csv('relabel_Muhammed.csv', **einstellungen)
teil_2.to_csv('relabel_Maryam.csv', **einstellungen)
teil_3.to_csv('relabel_Janina.csv', **einstellungen)

print("\n--- Aufteilung abgeschlossen ---")
print(f"Teil 1 gespeichert mit {len(teil_1)} Dokumenten (relabel_team_teil_1.csv)")
print(f"Teil 2 gespeichert mit {len(teil_2)} Dokumenten (relabel_team_teil_2.csv)")
print(f"Teil 3 gespeichert mit {len(teil_3)} Dokumenten (relabel_team_teil_3.csv)")


--- Aufteilung abgeschlossen ---
Teil 1 gespeichert mit 31 Dokumenten (relabel_team_teil_1.csv)
Teil 2 gespeichert mit 31 Dokumenten (relabel_team_teil_2.csv)
Teil 3 gespeichert mit 30 Dokumenten (relabel_team_teil_3.csv)


C:\Users\muham\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
